In [0]:
# ==========================================================
# 06_gold_indicador_brasil
# Camada: Gold
# Objetivo: agregar para o nível nacional
# ==========================================================

from pyspark.sql import functions as F

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
SCHEMA   = "default"

T_GOLD_MUNICIPIO = f"{CATALOGO}.{SCHEMA}.gold_indicador_municipio"
T_GOLD_UF        = f"{CATALOGO}.{SCHEMA}.gold_indicador_uf"
T_GOLD_ALUNOS    = f"{CATALOGO}.{SCHEMA}.gold_alunos"
T_GOLD_BRASIL    = f"{CATALOGO}.{SCHEMA}.gold_indicador_brasil"

print(f"Município : {T_GOLD_MUNICIPIO}")
print(f"UF        : {T_GOLD_UF}")
print(f"Alunos    : {T_GOLD_ALUNOS}")
print(f"Brasil    : {T_GOLD_BRASIL}")

In [0]:
# ==========================================================
# Célula 2 — Agregação nacional a partir da gold_indicador_uf
# ==========================================================

df_uf = spark.table(T_GOLD_UF)

print("=" * 70)
print("AGREGAÇÃO NACIONAL")
print("=" * 70)

# 1) Inspecionar o que existe por ano antes de agregar
print("\n[1] Base de entrada por ano:")
(df_uf
    .groupBy("ano")
    .agg(
        F.count("*").alias("ufs"),
        F.sum(F.when(F.col("taxa_observada").isNotNull(), 1).otherwise(0)).alias("com_taxa"),
        F.sum(F.when(F.col("meta_do_ano").isNotNull(), 1).otherwise(0)).alias("com_meta"),
        F.sum("qtd_redes").alias("redes_total"),
    )
    .orderBy("ano")
    .show(truncate=False))

# 2) Taxa nacional ponderada por qtd_redes
df_brasil = (df_uf
    .filter(F.col("taxa_observada").isNotNull())
    .groupBy("ano")
    .agg(
        F.sum("qtd_redes").alias("qtd_redes"),
        F.sum(F.col("taxa_observada") * F.col("qtd_redes")).alias("taxa_ponderada"),
        F.count("*").alias("ufs_com_dado"),
    )
    .withColumn("taxa_observada",
        F.round(F.col("taxa_ponderada") / F.col("qtd_redes"), 4))
    .drop("taxa_ponderada")
    .orderBy("ano"))

print("\n[2] Taxa nacional ponderada por qtd_redes:")
df_brasil.show(truncate=False)

print("=" * 70)

In [0]:
# ==========================================================
# Célula 3 — Taxa nacional a partir da gold_alunos
# Peso correto: qtd_alunos (não qtd_redes)
# ==========================================================

df_aluno = spark.table(T_GOLD_ALUNOS)

print("=" * 70)
print("TAXA NACIONAL — BASE gold_alunos")
print("=" * 70)

df_brasil = (df_aluno
    .groupBy("ano")
    .agg(
        F.sum("qtd_alunos").alias("qtd_alunos"),
        F.sum("qtd_escolas").alias("qtd_escolas"),
        F.sum(F.col("taxa_alfabetizacao") * F.col("qtd_alunos")).alias("taxa_ponderada"),
        F.countDistinct("id_municipio").alias("municipios"),
    )
    .withColumn("taxa_observada",
        F.round(F.col("taxa_ponderada") / F.col("qtd_alunos"), 4))
    .drop("taxa_ponderada")
    .orderBy("ano"))

df_brasil.show(truncate=False)

print("\nComparação com a ponderação por redes (gold_indicador_uf):")
print("  2023: alunos=57,89  |  redes=55,32")
print("  2024: alunos=59,57  |  redes=57,36")
print("  -> adotamos a ponderação por alunos (peso correto)")
print("=" * 70)

In [0]:
# ==========================================================
# Célula 4 — Montar a gold_indicador_brasil completa
# Taxa nacional + metas + gap + status
# ==========================================================

df_aluno = spark.table(T_GOLD_ALUNOS)
df_uf    = spark.table(T_GOLD_UF)

print("=" * 70)
print("MONTAGEM DA GOLD BRASIL")
print("=" * 70)

# 1) Taxa nacional por ano
brasil = (df_aluno
    .groupBy("ano")
    .agg(
        F.sum("qtd_alunos").alias("qtd_alunos"),
        F.sum("qtd_escolas").alias("qtd_escolas"),
        F.countDistinct("id_municipio").alias("qtd_municipios"),
        F.sum(F.col("taxa_alfabetizacao") * F.col("qtd_alunos")).alias("taxa_ponderada"),
    )
    .withColumn("taxa_observada",
        F.round(F.col("taxa_ponderada") / F.col("qtd_alunos"), 4))
    .drop("taxa_ponderada"))

# 2) Metas nacionais: pegar a meta_do_ano de cada UF e tirar a média
#    (a meta nacional é a mesma para todas as UFs em cada ano)
metas = (df_uf
    .filter(F.col("meta_do_ano").isNotNull())
    .groupBy("ano")
    .agg(F.round(F.avg("meta_do_ano"), 2).alias("meta_do_ano")))

# 3) Metas futuras (2024 a 2030) — pegar de qualquer UF, são iguais
metas_futuras = (df_uf
    .filter(F.col("meta_2024").isNotNull())
    .select("meta_2024", "meta_2025", "meta_2026", "meta_2027",
            "meta_2028", "meta_2029", "meta_2030")
    .limit(1))

print("\n[1] Metas nacionais por ano:")
metas.orderBy("ano").show(truncate=False)

print("\n[2] Metas futuras (amostra):")
metas_futuras.show(truncate=False)

print("=" * 70)

In [0]:
# ==========================================================
# Célula 5 — Meta nacional ponderada por percentual_participacao
# A gold_indicador_uf já tem meta_do_ano e o peso de cada UF
# ==========================================================

df_uf = spark.table(T_GOLD_UF)

print("=" * 70)
print("META NACIONAL PONDERADA")
print("=" * 70)

# 1) Conferir se percentual_participacao soma ~100 por ano
print("\n[1] Soma de percentual_participacao por ano:")
(df_uf
    .groupBy("ano")
    .agg(
        F.count("*").alias("ufs"),
        F.round(F.sum("percentual_participacao"), 2).alias("soma_pct"),
        F.sum(F.when(F.col("meta_do_ano").isNotNull(), 1).otherwise(0)).alias("com_meta"),
    )
    .orderBy("ano")
    .show(truncate=False))

# 2) Meta nacional ponderada
meta_pond = (df_uf
    .filter(F.col("meta_do_ano").isNotNull())
    .groupBy("ano")
    .agg(
        F.sum("percentual_participacao").alias("peso_total"),
        F.sum(F.col("meta_do_ano") * F.col("percentual_participacao")).alias("meta_ponderada"),
        F.count("*").alias("ufs_com_meta"),
    )
    .withColumn("meta_nacional",
        F.round(F.col("meta_ponderada") / F.col("peso_total"), 2))
    .select("ano", "ufs_com_meta", "peso_total", "meta_nacional")
    .orderBy("ano"))

print("\n[2] Meta nacional ponderada:")
meta_pond.show(truncate=False)

print("\n[3] Comparação:")
print("  Média simples das UFs : 2024=58,23 | 2025=62,23")
print("  Meta de uma UF (SP)   : 2024=59,20 | 2025=63,20")
print("=" * 70)

In [0]:
# ==========================================================
# Célula 6 — Montar e gravar a gold_indicador_brasil
# Taxa nacional da gold_alunos + meta nacional (média das UFs)
# ==========================================================

df_aluno = spark.table(T_GOLD_ALUNOS)
df_uf    = spark.table(T_GOLD_UF)

print("=" * 70)
print("MONTAGEM FINAL — gold_indicador_brasil")
print("=" * 70)

# 1) Taxa nacional por ano (fonte: gold_alunos)
brasil = (df_aluno
    .groupBy("ano")
    .agg(
        F.sum("qtd_alunos").alias("qtd_alunos"),
        F.sum("qtd_escolas").alias("qtd_escolas"),
        F.countDistinct("id_municipio").alias("qtd_municipios"),
        F.sum(F.col("taxa_alfabetizacao") * F.col("qtd_alunos")).alias("taxa_ponderada"),
    )
    .withColumn("taxa_observada",
        F.round(F.col("taxa_ponderada") / F.col("qtd_alunos"), 4))
    .drop("taxa_ponderada"))

# 2) Meta nacional = média simples das metas das UFs
metas = (df_uf
    .filter(F.col("meta_do_ano").isNotNull())
    .groupBy("ano")
    .agg(
        F.round(F.avg("meta_do_ano"), 2).alias("meta_do_ano"),
        F.count("*").alias("ufs_com_meta"),
    ))

# 3) Metas futuras (2024-2030): média das UFs por coluna
metas_futuras = (df_uf
    .filter(F.col("meta_2024").isNotNull())
    .agg(*[F.round(F.avg(c), 2).alias(c) for c in
           ["meta_2024","meta_2025","meta_2026","meta_2027",
            "meta_2028","meta_2029","meta_2030"]]))

# 4) Juntar tudo
df_brasil = (brasil
    .join(metas, on="ano", how="left")
    .crossJoin(metas_futuras)
    .withColumn("gap_pp",
        F.when(F.col("meta_do_ano").isNotNull(),
               F.round(F.col("taxa_observada") - F.col("meta_do_ano"), 2)))
    .withColumn("status_meta",
        F.when(F.col("taxa_observada").isNull(), "sem_dado")
         .when(F.col("meta_do_ano").isNull(), "sem_meta")
         .when(F.col("gap_pp") >= 0, "atingiu")
         .otherwise("abaixo"))
    .withColumn("tem_dado_completo",
        F.col("taxa_observada").isNotNull() & F.col("meta_do_ano").isNotNull())
    .select(
        "ano", "taxa_observada", "qtd_alunos", "qtd_escolas", "qtd_municipios",
        "meta_do_ano", "meta_2024", "meta_2025", "meta_2026", "meta_2027",
        "meta_2028", "meta_2029", "meta_2030",
        "gap_pp", "status_meta", "tem_dado_completo",
    )
    .orderBy("ano"))

print("\n[1] Resultado final:")
df_brasil.show(truncate=False)

print("\n[2] Schema:")
df_brasil.printSchema()

print("=" * 70)

In [0]:
# ==========================================================
# Célula 7 — Gravar a gold_indicador_brasil
# ==========================================================

print("=" * 70)
print("GRAVAÇÃO — gold_indicador_brasil")
print("=" * 70)

(df_brasil
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(T_GOLD_BRASIL))

print(f"\nTabela gravada: {T_GOLD_BRASIL}")

# Validação pós-gravação
df_check = spark.table(T_GOLD_BRASIL)
print(f"Registros: {df_check.count()}")
print(f"Colunas  : {len(df_check.columns)}")

print("\nConteúdo gravado:")
df_check.orderBy("ano").show(truncate=False)

print("=" * 70)